# Unified spaLLM Multi-Seed Workflow (Kaggle & Local Standalone Version)
This notebook integrates the joint spatial representation learning workflow for all 6 target datasets:
- **Mouse Brain E11, E13, E15, E18** (RNA + ATAC/epigenome)
- **Human Lymph Node A1, D1** (RNA + Protein/ADT)

## Usage Instructions:
1. **Sequential Execution**: Choose the datasets you wish to execute by updating the `ACTIVE_DATASETS` list in the Run Configuration cell.
2. **Running All**: Setting `ACTIVE_DATASETS = ["all"]` automatically sequences through all 6 datasets.
3. **Multi-Seed Averaging**: Set `SEEDS` to run each workflow across multiple random initialization seeds (default 10 seeds). Performance metrics (ARI, NMI, AMI, etc.) are compiled and averaged at the end of each dataset run.
4. **Plot Optimization**: To prevent notebook bloat, plots (UMAPs, Loss curves, Attention violins) are only rendered for the **first seed** run of each dataset.

In [1]:
# 1. Environment Setup (Kaggle & Colab compatible)
# Aligns 100% with Kaggle's stable pre-installed omics stack (anndata 0.10.x, scanpy 1.10.x)
# to eliminate dependency conflicts and kernel restarts completely.

import sys
import subprocess

def is_installed(package):
    try:
        __import__(package)
        return True
    except ImportError:
        return False

missing = []
if not is_installed("anndata"): missing.append("anndata==0.10.7")
if not is_installed("scanpy"): missing.append("scanpy==1.10.1")
if not is_installed("datasets"): missing.append("datasets")
if not is_installed("scgpt"): missing.append("scgpt==0.2.4")
if not is_installed("leidenalg"): missing.append("leidenalg")
if not is_installed("igraph"): missing.append("igraph")
if not is_installed("skmisc"): missing.append("scikit-misc")
if not is_installed("torchvision"): missing.append("torchvision")
if not is_installed("torchaudio"): missing.append("torchaudio")
if not is_installed("torchtext"): missing.append("torchtext")

if missing:
    print(f"Installing Kaggle-compatible packages: {missing}...")
    if "scgpt==0.2.4" in missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scgpt==0.2.4", "--no-deps"])
        missing.remove("scgpt==0.2.4")
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + missing)
    print("Installations completed successfully!")
else:
    print("All required packages are already installed and compatible.")


Installing Kaggle-compatible packages: ['anndata==0.10.7', 'scanpy==1.10.1', 'scgpt==0.2.4', 'leidenalg', 'scikit-misc', 'torchtext']...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 831.7/831.7 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.4/122.4 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 89.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.0/183.0 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.6/87.6 kB 6.7 MB/s eta 0:00:00
Installations completed successfully!


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
scgpt 0.2.4 requires cell-gears<0.0.3, which is not installed.
scgpt 0.2.4 requires orbax<0.1.8, which is not installed.
scgpt 0.2.4 requires scib<2.0.0,>=1.0.3, which is not installed.
scgpt 0.2.4 requires scvi-tools<1.0,>=0.16.0, which is not installed.
scgpt 0.2.4 requires datasets<3.0.0,>=2.3.0, but you have datasets 5.0.0 which is incompatible.


In [2]:
# ==================================================================
# RUN CONFIGURATION
# ==================================================================
# Choose environment mode: "auto" (detect Kaggle vs Local), "kaggle", or "local"
ENV_MODE = "auto"

# Define the configurations for all datasets
ALL_DATASETS_CONFIG = {
    "mouse-brain-e11-s1": {
        "type": "mouse_brain",
        "kaggle_dir": "/kaggle/input/datasets/sadmanbiazidarnob/mouse-brain-e11-s1/",
        "local_dir": "D:/FYDP/spaLLM/spaLLM/Data_SpatialGlue/Data_SpatialGlue/Dataset7_Mouse_Brain_ATAC/",
        "mod2_candidates": ["adata_ATAC.h5ad", "adata_peaks_normalized.h5ad"],
        "anno_file": "anno.csv"
    },
    "mouse-brain-e13-s1": {
        "type": "mouse_brain",
        "kaggle_dir": "/kaggle/input/datasets/sadmanbiazidarnob/mouse-brain-e13-s1/",
        "local_dir": "D:/FYDP/spaLLM/spaLLM/Data_SpatialGlue/Data_SpatialGlue/Dataset7_Mouse_Brain_ATAC/",
        "mod2_candidates": ["adata_ATAC.h5ad", "adata_peaks_normalized.h5ad"],
        "anno_file": "anno.csv"
    },
    "mouse-brain-e15-s1": {
        "type": "mouse_brain",
        "kaggle_dir": "/kaggle/input/datasets/sadmanbiazidarnob/mouse-brain-e15-s1/",
        "local_dir": "D:/FYDP/spaLLM/spaLLM/Data_SpatialGlue/Data_SpatialGlue/Dataset7_Mouse_Brain_ATAC/",
        "mod2_candidates": ["adata_ATAC.h5ad", "adata_peaks_normalized.h5ad"],
        "anno_file": "anno.csv"
    },
    "mouse-brain-e18-s1": {
        "type": "mouse_brain",
        "kaggle_dir": "/kaggle/input/datasets/sadmanbiazidarnob/mouse-brain-e18-s1/",
        "local_dir": "D:/FYDP/spaLLM/spaLLM/Data_SpatialGlue/Data_SpatialGlue/Dataset7_Mouse_Brain_ATAC/",
        "mod2_candidates": ["adata_ATAC.h5ad", "adata_peaks_normalized.h5ad"],
        "anno_file": "anno.csv"
    },
    "human-lymph-node-a1": {
        "type": "human_lymph_node",
        "kaggle_dir": "/kaggle/input/datasets/sadmanbiazidarnob/lymph-node-data/Dataset11_Human_Lymph_Node_A1/",
        "local_dir": "D:/FYDP/spaLLM/spaLLM/Data_SpatialGlue/Data_SpatialGlue/Dataset11_Human_Lymph_Node_A1/",
        "mod2_candidates": ["adata_ADT.h5ad"],
        "anno_file": "annotation.csv"
    },
    "human-lymph-node-d1": {
        "type": "human_lymph_node",
        "kaggle_dir": "/kaggle/input/datasets/sadmanbiazidarnob/human-lymph-node-d1/10x_human_lymph_node_D1/",
        "local_dir": "D:/FYDP/spaLLM/spaLLM/Data_SpatialGlue/Data_SpatialGlue/Dataset12_Human_Lymph_Node_D1/",
        "mod2_candidates": ["adata_ADT.h5ad"],
        "anno_file": "annotation.csv"
    }
}

# Select active datasets to execute sequentially:
# e.g. ["human-lymph-node-d1"], or list multiple names, or ["all"] to run everything sequentially
ACTIVE_DATASETS = ["all"]
import numpy as np
# Define high-entropy, statistically independent seeds for multi-seed ablation studies
MASTER_SEED = 42
rng = np.random.default_rng(MASTER_SEED)
SEEDS = rng.integers(low=1, high=2**31 - 1, size=10).tolist()

# Process "all" keyword option
if len(ACTIVE_DATASETS) == 1 and ACTIVE_DATASETS[0].lower() == "all":
    datasets_to_run = list(ALL_DATASETS_CONFIG.keys())
else:
    datasets_to_run = [d for d in ACTIVE_DATASETS if d in ALL_DATASETS_CONFIG]

print(f"Scheduled datasets: {datasets_to_run}")
print(f"Ablation seeds: {SEEDS}")


Scheduled datasets: ['mouse-brain-e11-s1', 'mouse-brain-e13-s1', 'mouse-brain-e15-s1', 'mouse-brain-e18-s1', 'human-lymph-node-a1', 'human-lymph-node-d1']
Ablation seeds: [191664964, 1662057957, 1405681632, 942484272, 929893138, 1843824992, 184566855, 1497586438, 432652534, 202244315]


## spaLLM Source Code
This cell contains the integrated spaLLM model architecture, graph preprocessing pipelines, and training wrappers.

In [3]:
# --- COMBINED spaLLM SOURCE CODE ---
import os
import random
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
# Inject pure-Python torchtext.vocab fallback mock if C++ libtorchtext.so is missing or broken
import types
class _PurePythonVocab:
    def __init__(self, vocab_dict=None, unk_token="<unk>"):
        if isinstance(vocab_dict, dict):
            self.stoi = dict(vocab_dict)
            self.itos = {v: k for k, v in vocab_dict.items()}
        elif isinstance(vocab_dict, (list, tuple)):
            self.itos = list(vocab_dict)
            self.stoi = {k: i for i, k in enumerate(vocab_dict)}
        else:
            self.stoi = {}
            self.itos = {}
        self.unk_token = unk_token
    def __getitem__(self, token):
        if token in self.stoi: return self.stoi[token]
        return self.stoi.get(self.unk_token, 0)
    def __len__(self): return len(self.stoi)
    def __contains__(self, token): return token in self.stoi
    def get_stoi(self): return self.stoi
    def get_itos(self): return self.itos
    def append_token(self, token):
        if token not in self.stoi:
            idx = len(self.itos)
            self.stoi[token] = idx
            self.itos[idx] = token
try:
    import torchtext
    import torchtext.vocab
except (ImportError, OSError):
    tt_mod = types.ModuleType("torchtext")
    tt_vocab_mod = types.ModuleType("torchtext.vocab")
    tt_vocab_mod.Vocab = _PurePythonVocab
    tt_vocab_mod.vocab = lambda dictionary: _PurePythonVocab(dictionary)
    tt_mod.vocab = tt_vocab_mod
    sys.modules["torchtext"] = tt_mod
    sys.modules["torchtext.vocab"] = tt_vocab_mod
import torch.nn.functional as F
from torch.nn.parameter import Parameter
from torch.nn.modules.module import Module
from torch.backends import cudnn
import sklearn
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors, kneighbors_graph
from scipy.sparse import coo_matrix
import anndata as ad
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
from typing import Optional

# 1. modelTriatt_Flow1.py
def init_weights(*params):
    """Initialize weights with Xavier uniform distribution."""
    for param in params:
        torch.nn.init.xavier_uniform_(param)

class DeepEncoder(Module):
    """Modality-specific GNN encoder."""
    def __init__(self, in_feat, out_feat, dropout=0.0, act=F.relu):
        super().__init__()
        self.dropout = dropout
        self.act = act
        self.hidden_dim = out_feat * 2

        self.weights = torch.nn.ParameterList([
            Parameter(torch.FloatTensor(in_feat, self.hidden_dim)),
            Parameter(torch.FloatTensor(self.hidden_dim, self.hidden_dim)),
            Parameter(torch.FloatTensor(self.hidden_dim, out_feat))
        ])
        init_weights(*self.weights)

    def forward(self, feat, adj):
        x = self._apply_layer(feat, adj, self.weights[0])
        x = self._apply_layer(x, adj, self.weights[1])
        x = torch.spmm(adj, torch.mm(x, self.weights[2]))
        return x

    def _apply_layer(self, x, adj, weight):
        x = torch.spmm(adj, torch.mm(x, weight))
        x = self.act(x)
        return F.dropout(x, self.dropout, training=self.training)

class CellEmbedding(Module):
    """Modality-specific cell embedding encoder/decoder."""
    def __init__(self, in_feat, out_feat):
        super().__init__()
        self.weight = Parameter(torch.FloatTensor(in_feat, out_feat))
        init_weights(self.weight)

    def forward(self, feat, adj):
        return torch.spmm(adj, torch.mm(feat, self.weight))

class AttentionLayer(Module):
    """Generic Attention Layer."""
    def __init__(self, in_feat, out_feat):
        super().__init__()
        self.w_omega = Parameter(torch.FloatTensor(in_feat, out_feat))
        self.u_omega = Parameter(torch.FloatTensor(out_feat, 1))
        init_weights(self.w_omega, self.u_omega)

    def forward(self, *embeddings):
        emb_stack = torch.cat([torch.unsqueeze(emb, dim=1) for emb in embeddings], dim=1)
        v = torch.tanh(torch.matmul(emb_stack, self.w_omega))
        vu = torch.matmul(v, self.u_omega)
        alpha = F.softmax(vu.squeeze(-1) + 1e-6, dim=1)
        emb_combined = torch.matmul(emb_stack.transpose(1, 2), alpha.unsqueeze(-1)).squeeze(-1)
        return emb_combined, alpha

class EncodingNetwork(Module):
    """Encoding network with modality-specific encoders, decoders, and attention layers."""
    def __init__(self, dim_in_omics1, dim_out_omics1, dim_in_omics2, dim_out_omics2):
        super().__init__()
        self.encoder_embedding = CellEmbedding(512, 64)
        self.decoder_embedding = CellEmbedding(64, 512)

        self.encoder_omics1 = DeepEncoder(dim_in_omics1, dim_out_omics1)
        self.decoder_omics1 = DeepEncoder(dim_out_omics1, dim_in_omics1)
        self.encoder_omics2 = DeepEncoder(dim_in_omics2, dim_out_omics2)
        self.decoder_omics2 = DeepEncoder(dim_out_omics2, dim_in_omics2)

        self.atten_feature1 = AttentionLayer(dim_out_omics1, dim_out_omics1)
        self.atten_feature2 = AttentionLayer(dim_out_omics1, dim_out_omics1)
        self.atten_feature = AttentionLayer(dim_out_omics1, dim_out_omics1)
        self.atten_omics2 = AttentionLayer(dim_out_omics2, dim_out_omics2)
        self.atten_cross = AttentionLayer(dim_out_omics1, dim_out_omics2)

    def forward(self, f_omics1, f_omics2, adj_spa1, adj_fea1, adj_spa2, adj_fea2, cell_emb, adj_emb):
        emb_spa = self.encoder_embedding(cell_emb, adj_spa1)
        emb_fea = self.encoder_embedding(cell_emb, adj_emb)

        emb_latent_spa1 = self.encoder_omics1(f_omics1, adj_spa1)
        emb_latent_spa2 = self.encoder_omics2(f_omics2, adj_spa2)
        emb_latent_fea1 = self.encoder_omics1(f_omics1, adj_fea1)
        emb_latent_fea2 = self.encoder_omics2(f_omics2, adj_fea2)

        emb_att1, alpha_att1 = self.atten_feature1(emb_spa, emb_latent_spa1)
        emb_att2, alpha_att2 = self.atten_feature2(emb_fea, emb_latent_fea1)
        emb_latent_omics1, alpha_att_omics1 = self.atten_feature(emb_att1, emb_att2)
        emb_latent_omics2, alpha_omics2 = self.atten_omics2(emb_latent_spa2, emb_latent_fea2)

        emb_latent_combined, alpha = self.atten_cross(emb_latent_omics1, emb_latent_omics2)

        emb_recon1 = self.decoder_omics1(emb_latent_combined, adj_spa1)
        emb_recon2 = self.decoder_omics2(emb_latent_combined, adj_spa2)
        emb_recon_spa = self.decoder_embedding(emb_spa, adj_spa1)
        emb_recon_fea = self.decoder_embedding(emb_fea, adj_emb)

        emb_cross1 = self.encoder_omics2(self.decoder_omics2(emb_latent_omics1, adj_spa2), adj_spa2)
        emb_cross2 = self.encoder_omics1(self.decoder_omics1(emb_latent_omics2, adj_spa1), adj_spa1)

        return {
            'emb_latent_omics1': emb_latent_omics1, 'emb_latent_omics2': emb_latent_omics2,
            'emb_latent_combined': emb_latent_combined, 'emb_recon_omics1': emb_recon1, 'emb_recon_omics2': emb_recon2,
            'emb_cross1': emb_cross1, 'emb_cross2': emb_cross2,
            'alpha_att1': alpha_att1, 'alpha_att2': alpha_att2, 'alpha_omics1': alpha_att_omics1,
            'alpha_omics2': alpha_omics2, 'alpha': alpha, 'emb_recon_spa': emb_recon_spa,
            'emb_recon_fea': emb_recon_fea
        }

# 2. preprocess.py
def fix_seed(seed: int):
    """Fix random seed for reproducibility."""
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    cudnn.deterministic = True
    cudnn.benchmark = False

def construct_neighbor_graph(adata_omics1, adata_omics2, datatype='SPOTS', n_neighbors=3):
    """Construct spatial and feature neighbor graphs."""
    if datatype == 'Spatial-epigenome-transcriptome':
        n_neighbors = 6

    def _construct_spatial_graph(adata):
        cell_position = adata.obsm['spatial']
        return construct_graph_by_coordinate(cell_position, n_neighbors)

    adata_omics1.uns['adj_spatial'] = _construct_spatial_graph(adata_omics1)
    adata_omics2.uns['adj_spatial'] = _construct_spatial_graph(adata_omics2)

    adata_omics1.obsm['adj_feature'], adata_omics2.obsm['adj_feature'] = construct_graph_by_feature(
        adata_omics1, adata_omics2
    )
    return {'adata_omics1': adata_omics1, 'adata_omics2': adata_omics2}

def pca(adata: ad.AnnData, use_reps=None, n_comps=10):
    """Perform PCA for dimensionality reduction."""
    pca_model = PCA(n_components=n_comps)
    data = adata.obsm[use_reps] if use_reps else adata.X
    data = data.toarray() if sp.issparse(data) else data
    return pca_model.fit_transform(data)

def clr_normalize_each_cell(adata: ad.AnnData, inplace=True):
    """Normalize count vector for each cell using CLR normalization."""
    def seurat_clr(x):
        s = np.sum(np.log1p(x[x > 0]))
        exp = np.exp(s / len(x))
        return np.log1p(x / exp)

    if not inplace:
        adata = adata.copy()
    adata.X = np.apply_along_axis(
        seurat_clr, 1, adata.X.toarray() if sp.issparse(adata.X) else np.array(adata.X)
    )
    return adata

def construct_graph_by_feature(adata_omics1, adata_omics2, k=20, mode="connectivity", metric="correlation"):
    """Construct feature neighbor graphs based on expression profiles."""
    graph_omics1 = kneighbors_graph(adata_omics1.obsm['feat'], k, mode=mode, metric=metric, include_self=False)
    graph_omics2 = kneighbors_graph(adata_omics2.obsm['feat'], k, mode=mode, metric=metric, include_self=False)
    return graph_omics1, graph_omics2

def construct_graph_by_coordinate(cell_position, n_neighbors=3):
    """Construct spatial graph based on spatial coordinates."""
    nbrs = NearestNeighbors(n_neighbors=n_neighbors + 1).fit(cell_position)
    _, indices = nbrs.kneighbors(cell_position)
    x = indices[:, 0].repeat(n_neighbors)
    y = indices[:, 1:].flatten()
    return pd.DataFrame({'x': x, 'y': y, 'value': 1})

def transform_adjacent_matrix(adj_df):
    """Transform adjacency dataframe into sparse matrix."""
    n_spot = adj_df['x'].max() + 1
    return coo_matrix((adj_df['value'], (adj_df['x'], adj_df['y'])), shape=(n_spot, n_spot))

def preprocess_graph(adj):
    """Normalize adjacency matrix for GNN input."""
    adj = sp.coo_matrix(adj + sp.eye(adj.shape[0]))
    rowsum = np.array(adj.sum(1))
    degree_inv_sqrt = sp.diags(np.power(rowsum, -0.5).flatten())
    return sparse_mx_to_torch_sparse_tensor(adj.dot(degree_inv_sqrt).T.dot(degree_inv_sqrt))

def sparse_mx_to_torch_sparse_tensor(sparse_mx):
    """Convert scipy sparse matrix to torch sparse tensor."""
    sparse_mx = sparse_mx.tocoo().astype(np.float32)
    indices = torch.from_numpy(np.vstack((sparse_mx.row, sparse_mx.col)).astype(np.int64))
    return torch.sparse.FloatTensor(indices, torch.from_numpy(sparse_mx.data), torch.Size(sparse_mx.shape))

def adjacent_matrix_preprocessing(adata_omics1, adata_omics2, adj_emb):
    """Preprocess spatial and feature adjacency matrices for GNNs."""
    def _process_adj(adj):
        adj = adj.toarray() + adj.toarray().T
        adj = np.where(adj > 1, 1, adj)
        return preprocess_graph(adj)

    adj_spatial_omics1 = _process_adj(transform_adjacent_matrix(adata_omics1.uns['adj_spatial']))
    adj_spatial_omics2 = _process_adj(transform_adjacent_matrix(adata_omics2.uns['adj_spatial']))
    adj_emb = _process_adj(adj_emb)

    def _process_feature_adj(adj):
        adj = adj + adj.T
        return preprocess_graph(np.where(adj > 1, 1, adj))

    adj_feature_omics1 = _process_feature_adj(torch.FloatTensor(adata_omics1.obsm['adj_feature'].toarray()))
    adj_feature_omics2 = _process_feature_adj(torch.FloatTensor(adata_omics2.obsm['adj_feature'].toarray()))

    return {
        'adj_spatial_omics1': adj_spatial_omics1,
        'adj_spatial_omics2': adj_spatial_omics2,
        'adj_feature_omics1': adj_feature_omics1,
        'adj_feature_omics2': adj_feature_omics2,
        'adj_emb': adj_emb
    }

def lsi(adata: ad.AnnData, n_components: int = 20, use_highly_variable: Optional[bool] = None, **kwargs):
    """LSI analysis (Seurat v3 approach)."""
    use_highly_variable = use_highly_variable if use_highly_variable is not None else "highly_variable" in adata.var
    adata_use = adata[:, adata.var["highly_variable"]] if use_highly_variable else adata
    X_norm = sklearn.preprocessing.Normalizer(norm="l1").fit_transform(tfidf(adata_use.X))
    X_lsi = sklearn.utils.extmath.randomized_svd(np.log1p(X_norm * 1e4), n_components, **kwargs)[0]
    adata.obsm["X_lsi"] = (X_lsi - X_lsi.mean(axis=1, keepdims=True)) / X_lsi.std(axis=1, ddof=1, keepdims=True)

def tfidf(X):
    """TF-IDF normalization following Seurat v3 approach."""
    idf = X.shape[0] / X.sum(axis=0)
    tf = X.multiply(1 / X.sum(axis=1)) if sp.issparse(X) else X / X.sum(axis=1, keepdims=True)
    return tf.multiply(idf) if sp.issparse(X) else tf * idf

# 3. utils.py
try:
    import rpy2.robjects as robjects
    import rpy2.robjects.numpy2ri
    rpy2.robjects.numpy2ri.activate()
except Exception as e:
    robjects = None
    print(f"Warning: R environment not found or rpy2 error. R-based functionality like mclust will not be available.")

def convert_csv_to_h5ad(input_path, output_path):
    """Convert a CSV file to AnnData h5ad format."""
    data = pd.read_csv(input_path, index_col=0)
    obs_data = data.iloc[:, :2]
    obs_data.columns = ['barcode', 'assigned_cluster']
    expr_data = data.iloc[:, 2:].values.astype('float32')
    gene_names = data.columns[2:]
    adata = sc.AnnData(X=expr_data)
    adata.obs = obs_data
    adata.var['gene_names'] = gene_names
    adata.write(output_path)
    print(f"Data saved to {output_path}")

def convert_tsv_to_csv(tsv_path, csv_path):
    """Convert a TSV file to CSV format."""
    data = pd.read_csv(tsv_path, sep='\t')
    data.to_csv(csv_path, index=False)
    print(f"TSV file converted to CSV and saved at {csv_path}")

def create_h5ad_from_sparse_csv(input_csv, output_h5ad):
    """Create an AnnData h5ad file from a sparse matrix CSV file."""
    import scipy.sparse as sp
    import anndata as ad
    data = pd.read_csv(input_csv, index_col=0)
    expression_matrix = sp.csr_matrix(data.values)
    spatial_coords = data.index.str.split('x').tolist()
    spatial_coords = np.array(spatial_coords, dtype=int)
    adata = ad.AnnData(X=expression_matrix)
    adata.obsm['spatial'] = spatial_coords
    adata.var['gene_names'] = data.columns.values
    adata.write(output_h5ad)
    print(f"Sparse AnnData file saved to {output_h5ad}")

def mclust_R(adata, num_cluster, modelNames='EEE', used_obsm='emb_pca', random_seed=2020):
    """Perform clustering using the R `mclust` algorithm."""
    if robjects is None:
        raise ImportError("rpy2 or R environment not found. Ensure R is installed and rpy2 is correctly configured.")
    robjects.r.library("mclust")
    robjects.r['set.seed'](random_seed)
    res = robjects.r['Mclust'](rpy2.robjects.numpy2ri.numpy2rpy(adata.obsm[used_obsm]), num_cluster, modelNames)
    adata.obs['mclust'] = np.array(res[-2]).astype('int')
    return adata

def clustering(adata, n_clusters=7, key='emb', add_key='spaLLM', method='leiden', **kwargs):
    """Spatial clustering using `mclust`, `leiden`, or `louvain`."""
    use_pca = kwargs.get('use_pca', False)
    n_comps = kwargs.get('n_comps', 20)
    if use_pca:
        adata.obsm[key + '_pca'] = pca(adata, use_reps=key, n_comps=n_comps)
        key = key + '_pca'

    if method == 'mclust':
        mclust_R(adata, num_cluster=n_clusters, used_obsm=key)
        adata.obs[add_key] = adata.obs['mclust']
    if method in ['leiden', 'louvain']:
        search_kwargs = {k: v for k, v in kwargs.items() if k not in ['method', 'use_rep']}
        res = search_res(adata, n_clusters, method=method, use_rep=key, **search_kwargs)
        if method == 'leiden':
            try:
                sc.tl.leiden(adata, random_state=0, resolution=res, flavor='igraph', n_iterations=2, directed=False)
            except TypeError:
                sc.tl.leiden(adata, random_state=0, resolution=res)
        else:
            sc.tl.louvain(adata, random_state=0, resolution=res)
        adata.obs[add_key] = adata.obs[method].astype(int) if method in adata.obs else None

def add_noise_by_zeroing(matrix, zero_prob):
    """Add noise by zeroing random elements in the matrix."""
    noise_mask = torch.bernoulli((1 - zero_prob) * torch.ones_like(matrix)).to(matrix.device)
    return matrix * noise_mask

def add_noise_by_zeroing_columns(matrix, zero_prob=0.1):
    """Add noise by zeroing entire random columns of the matrix."""
    zero_mask = torch.bernoulli((1 - zero_prob) * torch.ones(matrix.size(1))).to(matrix.device)
    return matrix * zero_mask.unsqueeze(0).expand_as(matrix)

def add_gaussian_noise(matrix, mean=0.0, std=0.001):
    """Add Gaussian noise to the matrix."""
    noise = torch.normal(mean=mean, std=std, size=matrix.size()).to(matrix.device)
    return matrix + noise

def search_res(adata, n_clusters, method='leiden', use_rep='emb', start=0.1, end=3.0, increment=0.01, **kwargs):
    """Search for resolution to achieve target cluster count using `leiden` or `louvain`."""
    print('Searching resolution...')
    sc.pp.neighbors(adata, n_neighbors=50, use_rep=use_rep)
    for res in np.arange(start, end, increment):
        res = round(res, 3)
        if method == 'leiden':
            try:
                sc.tl.leiden(adata, random_state=0, resolution=res, flavor='igraph', n_iterations=2, directed=False)
            except TypeError:
                sc.tl.leiden(adata, random_state=0, resolution=res)
            clusters = adata.obs['leiden']
        elif method == 'louvain':
            sc.tl.louvain(adata, random_state=0, resolution=res)
            clusters = adata.obs['louvain']
        count_unique = clusters.nunique()
        print(f'resolution={res}, cluster number={count_unique}')
        if count_unique == n_clusters:
            return res
    raise ValueError("Resolution not found. Please try a larger range or smaller step size.")

# 4. spaLLM_util.py
class Train_spaLLM:
    def __init__(self, data, embedding, datatype='10x', device=torch.device('cpu'),
                 random_seed=2024, learning_rate=0.0001, weight_decay=0.0, epochs=600,
                 dim_input=3000, dim_output=64, weight_factors=None):
        self.device = device
        self.data = data.copy()
        self.embedding = torch.from_numpy(embedding).to(device)
        self.datatype = datatype
        self.random_seed = random_seed
        self.learning_rate = learning_rate
        self.weight_decay = weight_decay
        self.epochs = epochs
        self.dim_input = dim_input
        self.dim_output = dim_output
        self.weight_factors = weight_factors or [5, 5, 1, 10, 10, 10]

        self._init_adj_and_features()
        self.loss_history = []
        self._adjust_hyperparameters()

    def _init_adj_and_features(self):
        """Initialize adjacency matrices and input features."""
        adj = adjacent_matrix_preprocessing(self.data['adata_omics1'], self.data['adata_omics2'], self.data['adj_emb'])
        self.adj_spatial_omics1 = adj['adj_spatial_omics1'].to(self.device)
        self.adj_spatial_omics2 = adj['adj_spatial_omics2'].to(self.device)
        self.adj_feature_omics1 = adj['adj_feature_omics1'].to(self.device)
        self.adj_feature_omics2 = adj['adj_feature_omics2'].to(self.device)
        self.adj_emb = adj['adj_emb'].to(self.device)

        self.features_omics1 = torch.FloatTensor(self.data['adata_omics1'].obsm['feat']).to(self.device)
        self.features_omics2 = torch.FloatTensor(self.data['adata_omics2'].obsm['feat']).to(self.device)
        self.dim_input1, self.dim_input2 = self.features_omics1.shape[1], self.features_omics2.shape[1]

    def _adjust_hyperparameters(self):
        """Adjust hyperparameters based on data type."""
        if self.datatype == 'SPOTS':
            self.epochs, self.weight_factors = 600, [1, 5, 1, 1, 5, 5]
        elif self.datatype == '10x':
            self.epochs, self.weight_factors = 200, [5, 5, 1, 10, 10, 10]
        elif self.datatype == 'Spatial-epigenome-transcriptome':
            self.epochs, self.weight_factors = 1600, [1, 5, 1, 1, 10, 10]

    def _add_noise(self):
        """Apply Gaussian noise to features and embeddings."""
        features_omics1_noisy = add_gaussian_noise(self.features_omics1, mean=0, std=0.1)
        embedding_noisy = add_gaussian_noise(self.embedding, mean=0, std=0.01)
        return features_omics1_noisy, embedding_noisy

    def _calculate_losses(self, results):
        """Calculate reconstruction and correspondence losses."""
        loss_recon_omics1 = F.mse_loss(self.features_omics1, results['emb_recon_omics1'])
        loss_recon_omics2 = F.mse_loss(self.features_omics2, results['emb_recon_omics2'])
        loss_rec_es = F.mse_loss(self.embedding, results['emb_recon_spa'])
        loss_rec_ef = F.mse_loss(self.embedding, results['emb_recon_fea'])
        loss_corr_omics1 = F.mse_loss(results['emb_latent_omics1'], results['emb_cross1'])
        loss_corr_omics2 = F.mse_loss(results['emb_latent_omics2'], results['emb_cross2'])

        loss = (self.weight_factors[0] * loss_recon_omics1 +
                self.weight_factors[1] * loss_recon_omics2 +
                self.weight_factors[2] * loss_corr_omics1 +
                self.weight_factors[3] * loss_corr_omics2 +
                self.weight_factors[4] * loss_rec_es +
                self.weight_factors[5] * loss_rec_ef)
        return loss

    def train(self, epochs=None):
        epochs = epochs or self.epochs
        self.model = EncodingNetwork(self.dim_input1, self.dim_output, self.dim_input2, self.dim_output).to(self.device)
        optimizer = torch.optim.Adam(self.model.parameters(), lr=self.learning_rate, weight_decay=self.weight_decay)

        for epoch in range(epochs):
            optimizer.zero_grad()
            if random.random() < 0.5:
                features_omics1, embedding = self._add_noise()
            else:
                features_omics1, embedding = self.features_omics1, self.embedding

            results = self.model(features_omics1, self.features_omics2, self.adj_spatial_omics1,
                                 self.adj_feature_omics1,
                                 self.adj_spatial_omics2, self.adj_feature_omics2, embedding, self.adj_emb)
            loss = self._calculate_losses(results)
            loss.backward()
            optimizer.step()
            self.loss_history.append(loss.item())

        return self._evaluate_model()

    def _evaluate_model(self):
        """Evaluate the model and return output embeddings."""
        self.model.eval()
        with torch.no_grad():
            results = self.model(self.features_omics1, self.features_omics2, self.adj_spatial_omics1,
                                 self.adj_feature_omics1, self.adj_spatial_omics2, self.adj_feature_omics2,
                                 self.embedding, self.adj_emb)

        return {
            'emb_latent_omics1': F.normalize(results['emb_latent_omics1'], p=2).cpu().numpy(),
            'emb_latent_omics2': F.normalize(results['emb_latent_omics2'], p=2).cpu().numpy(),
            'spaLLM': F.normalize(results['emb_latent_combined'], p=2).cpu().numpy(),
            'alpha_omics1': results['alpha_omics1'].cpu().numpy(),
            'alpha_omics2': results['alpha_omics2'].cpu().numpy(),
            'alpha': results['alpha'].cpu().numpy(),
            'alpha_att1': results['alpha_att1'].cpu().numpy(),
            'alpha_att2': results['alpha_att2'].cpu().numpy()
        }

    def plot_loss(self):
        """Plot the training loss curve."""
        plt.figure(figsize=(10, 6))
        plt.plot(self.loss_history, label='Training Loss')
        plt.xlabel('Epochs')
        plt.ylabel('Loss')
        plt.title('Loss Curve')
        plt.legend()
        plt.show()


## Setup & Device Configuration

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device} (If this says 'cpu', make sure GPU is enabled!)")


Using device: cuda (If this says 'cpu', make sure GPU is enabled!)


## Encapsulated spaLLM Dataset Workflow
Defines the complete execution sequence for a single dataset: Loading, scGPT Embedding, Preprocessing, GNN Training, and UMAP / Metrics Evaluation.

In [5]:
def run_spallm_workflow(dataset_name, dataset_cfg, env_mode, seed, device, show_plots=False):
    print(f"\n--- Running Seed: {seed} ---")
    fix_seed(seed)
    
    # 1. Resolve paths
    is_kaggle = os.path.exists('/kaggle/input')
    if env_mode == "kaggle" or (env_mode == "auto" and is_kaggle):
        data_dir = dataset_cfg["kaggle_dir"]
        model_dir = '/kaggle/input/datasets/sadmanbiazidarnob/scgpt-human/scGPT_human'
    else:
        data_dir = dataset_cfg["local_dir"]
        model_dir = r"D:/FYDP/spaLLM/spaLLM/scGPT_human"
        
    dataset_type = dataset_cfg["type"]
    
    print(f"Loading data from: {data_dir}")
    adata_rna = sc.read_h5ad(os.path.join(data_dir, 'adata_RNA.h5ad'))
    
    # Find second modality file
    mod2_filename = None
    for cand in dataset_cfg["mod2_candidates"]:
        if os.path.exists(os.path.join(data_dir, cand)):
            mod2_filename = cand
            break
    if mod2_filename is None:
        mod2_filename = dataset_cfg["mod2_candidates"][0]  # fallback
        
    adata_mod2 = sc.read_h5ad(os.path.join(data_dir, mod2_filename))
    
    # Load cell type annotations if available, otherwise set to unknown
    annotation_loaded = False
    annotation_filename = dataset_cfg["anno_file"]
    annotation_path = os.path.join(data_dir, annotation_filename)
    
    if os.path.exists(annotation_path):
        print(f"Loading annotations from: {annotation_path}")
        annotation = pd.read_csv(annotation_path)
        
        # Unify columns for barcode
        for col in ['Barcode', 'barcode']:
            if col in annotation.columns:
                annotation = annotation.rename(columns={col: 'barcode'})
                break
                
        # Unify columns for ground truth annotation
        for col in ['manual-anno', 'cluster', 'ground_truth']:
            if col in annotation.columns:
                annotation = annotation.rename(columns={col: 'ground_truth'})
                break
                
        annotation = annotation.set_index('barcode')
        
        adata_rna.obs = adata_rna.obs.join(annotation, how='left')
        adata_rna.obs['ground_truth'] = adata_rna.obs['ground_truth'].fillna('unknown')
        
        adata_mod2.obs = adata_mod2.obs.join(annotation, how='left')
        adata_mod2.obs['ground_truth'] = adata_mod2.obs['ground_truth'].fillna('unknown')
        annotation_loaded = True
    else:
        print(f"Warning: Annotation file '{annotation_filename}' not found. Defaulting to 'unknown'.")
        adata_rna.obs['ground_truth'] = 'unknown'
        adata_mod2.obs['ground_truth'] = 'unknown'
        
    print("RNA shape:", adata_rna.shape)
    print("Modality 2 shape:", adata_mod2.shape)
    
    # 2. scGPT Embedding
    adata_rna.var_names_make_unique()
    adata_mod2.var_names_make_unique()
    adata_rna.var['gene_names'] = adata_rna.var.index.str.upper()
    
    if os.path.exists(model_dir):
        print(f"Running scGPT embedding on GPU using model at: {model_dir}")
        from scgpt.tasks.cell_emb import embed_data
        adata_emb = embed_data(
            adata_or_file=adata_rna.copy(), 
            model_dir=model_dir, 
            gene_col="gene_names", 
            max_length=1200, 
            batch_size=64, 
            obs_to_save=None, 
            device=device, 
            use_fast_transformer=False, 
            return_new_adata=False
        )
        embedding = adata_emb.obsm["X_scGPT"]
        print("scGPT embedding generated successfully.")
    else:
        print(f"Warning: scGPT model directory '{model_dir}' not found.")
        print("Generating dummy mock embeddings for execution consistency...")
        embedding = np.random.normal(size=(adata_rna.n_obs, 512))
        
    # 3. Preprocessing
    print("Preprocessing RNA data...")
    sc.pp.filter_genes(adata_rna, min_cells=10)
    sc.pp.highly_variable_genes(adata_rna, flavor="seurat_v3", n_top_genes=3000)
    sc.pp.normalize_total(adata_rna, target_sum=1e4)
    sc.pp.log1p(adata_rna)
    sc.pp.scale(adata_rna)
    
    adata_rna_high = adata_rna[:, adata_rna.var['highly_variable']]
    
    if dataset_type == "human_lymph_node":
        n_comps_rna = adata_mod2.n_vars - 1
        print(f"Using {n_comps_rna} components for RNA PCA (based on ADT n_vars)")
        adata_rna.obsm['feat'] = pca(adata_rna_high, n_comps=n_comps_rna)
        
        print("Preprocessing Protein (ADT) data...")
        adata_mod2 = clr_normalize_each_cell(adata_mod2)
        sc.pp.scale(adata_mod2)
        adata_mod2.obsm['feat'] = pca(adata_mod2, n_comps=adata_mod2.n_vars - 1)
    else:
        n_comps_rna = min(50, adata_rna_high.n_obs - 1, adata_rna_high.n_vars - 1)
        print(f"Using {n_comps_rna} components for RNA PCA")
        adata_rna.obsm['feat'] = pca(adata_rna_high, n_comps=n_comps_rna)
        
        print("Preprocessing Epigenome (ATAC) data...")
        sc.pp.normalize_total(adata_mod2, target_sum=1e4)
        sc.pp.log1p(adata_mod2)
        sc.pp.scale(adata_mod2)
        n_comps_mod2 = min(50, adata_mod2.n_obs - 1, adata_mod2.n_vars - 1)
        print(f"Using {n_comps_mod2} components for ATAC PCA")
        adata_mod2.obsm['feat'] = pca(adata_mod2, n_comps=n_comps_mod2)
        
    # 4. Training
    print("Constructing neighbor graphs...")
    data = construct_neighbor_graph(adata_rna, adata_mod2, datatype='10x')
    data['adj_emb'] = kneighbors_graph(embedding, 20, mode="connectivity", metric="correlation", include_self=False)
    
    print("Training spaLLM...")
    model = Train_spaLLM(data, datatype='10x', device=device, embedding=embedding)
    output = model.train(epochs=800)
    
    adata = adata_rna.copy()
    for key, value in output.items():
        adata.obsm[key] = value
    print("Training complete.")
    
    # 5. Evaluation and Spatial Plotting
    from sklearn.metrics import (
        adjusted_rand_score,
        normalized_mutual_info_score,
        adjusted_mutual_info_score,
        homogeneity_score,
        v_measure_score,
        silhouette_score
    )
    
    valid_mask = (adata.obs['ground_truth'] != 'unknown') & (adata.obs['ground_truth'] != 'Exclude')
    valid_labels = adata.obs['ground_truth'][valid_mask]
    
    if len(valid_labels) > 0:
        n_clusters = len(np.unique(valid_labels))
        print(f"Clustering into {n_clusters} clusters (based on ground truth)...")
    else:
        n_clusters = 7
        print(f"No valid ground truth annotations. Clustering into default {n_clusters} clusters...")
        
    clustering(adata, key='spaLLM', add_key='spaLLM', n_clusters=n_clusters, method='leiden', use_pca=True)
    
    metrics = None
    if len(valid_labels) > 0:
        y_true = adata.obs['ground_truth'][valid_mask].astype(str)
        y_pred = adata.obs['spaLLM'][valid_mask].astype(str)
        feats = adata.obsm['spaLLM'][valid_mask]
        
        ari = adjusted_rand_score(y_true, y_pred)
        nmi = normalized_mutual_info_score(y_true, y_pred)
        ami = adjusted_mutual_info_score(y_true, y_pred)
        homogeneity = homogeneity_score(y_true, y_pred)
        v_measure = v_measure_score(y_true, y_pred)
        y_pred_encoded = y_pred.astype(int) if y_pred.str.isdigit().all() else pd.factorize(y_pred)[0]
        sil = silhouette_score(feats, y_pred_encoded)
        
        print(f"\n=== {dataset_name} Performance (Seed {seed}) ===")
        print(f"ARI: {ari:.4f}")
        print(f"NMI: {nmi:.4f}")
        print(f"AMI: {ami:.4f}")
        print(f"Homogeneity: {homogeneity:.4f}")
        print(f"V-measure: {v_measure:.4f}")
        print(f"Silhouette: {sil:.4f}")
        
        metrics = {
            "ARI": ari,
            "NMI": nmi,
            "AMI": ami,
            "Homogeneity": homogeneity,
            "V-measure": v_measure,
            "Silhouette": sil
        }
    else:
        print("\nSkipping evaluation metrics (no ground truth annotation available).")
        
    if show_plots:
        # Compute neighbors on spaLLM embedding for UMAP projection
        sc.pp.neighbors(adata, use_rep='spaLLM', n_neighbors=10)
        sc.tl.umap(adata)
        
        if len(valid_labels) > 0:
            fig, ax_list = plt.subplots(1, 3, figsize=(15, 4))
            sc.pl.umap(adata, color='spaLLM', ax=ax_list[0], title='spaLLM UMAP', show=False)
            sc.pl.embedding(adata, basis='spatial', color='spaLLM', ax=ax_list[1], title='spaLLM Spatial', show=False)
            sc.pl.embedding(adata, basis='spatial', color='ground_truth', ax=ax_list[2], title='Ground Truth Spatial', show=False)
        else:
            fig, ax_list = plt.subplots(1, 2, figsize=(10, 4))
            sc.pl.umap(adata, color='spaLLM', ax=ax_list[0], title='spaLLM UMAP', show=False)
            sc.pl.embedding(adata, basis='spatial', color='spaLLM', ax=ax_list[1], title='spaLLM Spatial', show=False)
        plt.tight_layout()
        plt.show()
        
        # Plots (Loss Curve and Violins)
        model.plot_loss()
        
        def plot_violin(adata, alpha_key, title):
            import pandas as pd
            import seaborn as sns
            import matplotlib.pyplot as plt
            
            plt.rcParams['figure.figsize'] = (8, 5)
            df = pd.DataFrame({
                'embedding': adata.obsm[alpha_key][:, 0],
                'omic': adata.obsm[alpha_key][:, 1],
                'label': adata.obs['spaLLM']
            })
            df = df.set_index('label').stack().reset_index()
            df.columns = ['Cluster_Label', 'Modality', 'Weight value']
            ax = sns.violinplot(data=df, x='Cluster_Label', y='Weight value', hue='Modality', split=True, inner='quart', linewidth=1)
            ax.set_title(title)
            ax.set_xlabel('spaLLM Cluster Label')
            ax.legend(bbox_to_anchor=(1.4, 1.01), loc='upper right')
            plt.tight_layout(w_pad=0.05)
            plt.show()
            
        plot_violin(adata, 'alpha_att1', 'Attention Weights: Spatial Embedding vs Omics')
        plot_violin(adata, 'alpha_att2', 'Attention Weights: Feature Embedding vs Omics')
        
    return metrics


## Execute Selected Datasets & Aggregate Results
Sequentially loops over all selected datasets and executes the spaLLM pipeline across all defined seeds, computing final averages and standard deviations for the metrics.

In [6]:
all_results_flat = []
all_results = {}

for dname in datasets_to_run:
    cfg = ALL_DATASETS_CONFIG[dname]
    all_results[dname] = []
    
    print(f"\n=======================================================")
    print(f"STARTING WORKFLOW FOR DATASET: {dname} OVER {len(SEEDS)} SEEDS")
    print(f"=======================================================")
    
    for idx, seed in enumerate(SEEDS):
        # Only show plots on the first seed run to avoid notebook bloating
        show_plots = (idx == 0)
        try:
            metrics = run_spallm_workflow(dname, cfg, ENV_MODE, seed, device, show_plots=show_plots)
            if metrics is not None:
                all_results[dname].append(metrics)
                # Save flat entry for CSV log
                row = {"dataset": dname, "seed": seed}
                row.update(metrics)
                all_results_flat.append(row)
        except Exception as e:
            print(f"Error processing dataset {dname} with seed {seed}: {e}")
            
    # Calculate and report average performance metrics
    if len(all_results[dname]) > 0:
        df_metrics = pd.DataFrame(all_results[dname])
        print(f"\n=======================================================")
        print(f"AVERAGE PERFORMANCE FOR {dname} ({len(all_results[dname])} successful runs)")
        print(f"=======================================================")
        print(df_metrics.mean().to_string())
        print("\nStandard Deviation:")
        print(df_metrics.std().to_string())
        print(f"=======================================================\n")
    else:
        print(f"No evaluation metrics collected for {dname}.")

# Save all collected metrics to CSV in Kaggle working directory
if len(all_results_flat) > 0:
    df_all = pd.DataFrame(all_results_flat)
    is_kaggle = os.path.exists('/kaggle/working')
    output_dir = '/kaggle/working' if is_kaggle else '.'
    output_csv = os.path.join(output_dir, 'spallm_ablation_results.csv')
    df_all.to_csv(output_csv, index=False)
    print(f"All ablation study results saved to CSV at: {output_csv}")
else:
    print("No results were generated, skipping CSV export.")



STARTING WORKFLOW FOR DATASET: mouse-brain-e11-s1 OVER 10 SEEDS

--- Running Seed: 191664964 ---
Loading data from: /kaggle/input/datasets/sadmanbiazidarnob/mouse-brain-e11-s1/


/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Loading annotations from: /kaggle/input/datasets/sadmanbiazidarnob/mouse-brain-e11-s1/anno.csv
RNA shape: (1263, 32285)
Modality 2 shape: (1263, 69370)
Running scGPT embedding on GPU using model at: /kaggle/input/datasets/sadmanbiazidarnob/scgpt-human/scGPT_human


/usr/local/lib/python3.12/dist-packages/scgpt/model/model.py:21: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")
/usr/local/lib/python3.12/dist-packages/scgpt/model/multiomic_model.py:19: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")


Error processing dataset mouse-brain-e11-s1 with seed 191664964: <lambda>() got an unexpected keyword argument 'min_freq'

--- Running Seed: 1662057957 ---
Loading data from: /kaggle/input/datasets/sadmanbiazidarnob/mouse-brain-e11-s1/


...

/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Loading annotations from: /kaggle/input/datasets/sadmanbiazidarnob/human-lymph-node-d1/10x_human_lymph_node_D1/annotation.csv
RNA shape: (3359, 18085)
Modality 2 shape: (3359, 31)
Running scGPT embedding on GPU using model at: /kaggle/input/datasets/sadmanbiazidarnob/scgpt-human/scGPT_human
Error processing dataset human-lymph-node-d1 with seed 1497586438: <lambda>() got an unexpected keyword argument 'min_freq'

--- Running Seed: 432652534 ---
Loading data from: /kaggle/input/datasets/sadmanbiazidarnob/human-lymph-node-d1/10x_human_lymph_node_D1/


/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Loading annotations from: /kaggle/input/datasets/sadmanbiazidarnob/human-lymph-node-d1/10x_human_lymph_node_D1/annotation.csv
RNA shape: (3359, 18085)
Modality 2 shape: (3359, 31)
Running scGPT embedding on GPU using model at: /kaggle/input/datasets/sadmanbiazidarnob/scgpt-human/scGPT_human
Error processing dataset human-lymph-node-d1 with seed 432652534: <lambda>() got an unexpected keyword argument 'min_freq'

--- Running Seed: 202244315 ---
Loading data from: /kaggle/input/datasets/sadmanbiazidarnob/human-lymph-node-d1/10x_human_lymph_node_D1/
Loading annotations from: /kaggle/input/datasets/sadmanbiazidarnob/human-lymph-node-d1/10x_human_lymph_node_D1/annotation.csv
RNA shape: (3359, 18085)
Modality 2 shape: (3359, 31)
Running scGPT embedding on GPU using model at: /kaggle/input/datasets/sadmanbiazidarnob/scgpt-human/scGPT_human
Error processing dataset human-lymph-node-d1 with seed 202244315: <lambda>() got an unexpected keyword argument 'min_freq'
No evaluation metrics collected 

/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
